# NB14 — Soil-Restricted Primary Replication

Tests whether the per-Mb metal gene specialization signal holds when niche breadth
is computed **within soil samples only** (8 Env_Level_1 categories) rather than across
all 13 environment types. This is a sensitivity check: if the primary finding is driven
by soil vs. non-soil contrasts it would disappear; if it reflects within-soil ecological
differentiation it should persist.

**Primary result to replicate:** Total 94-KO/Mb β=−0.022, p=4×10⁻⁷ (n=997);
Tier 2 homeostasis/Mb β=−0.009, p=0.011; Tier 1 p=0.256 (null).


## Status: REPLICATES (n=603 genera)

**Niche metric**: Levins' B_std restricted to 8 soil Env_Level_1 categories
(soil, agricultural, farm, field, paddy, peatland, desert, shrub).
Computed from `data/otu_env_matrix.csv` + `data/env_totals.csv`.

**Result** (n=603 genera after GTDB tree pruning from 628):

| Predictor | β | SE | p-value | λ |
|---|---|---|---|---|
| Total 94-KO/Mb (z) | **−0.023** | 0.0061 | **0.00020** | 0.770 |
| Tier 1 resistance/Mb (z) | −0.0064 | 0.0054 | 0.238 | 0.780 |
| Tier 2 homeostasis/Mb (z) | −0.0025 | 0.0052 | 0.626 | 0.780 |

**Interpretation**: The total 94-KO/Mb specialization signal replicates within soil alone
(β=−0.023 vs. primary β=−0.022). The primary finding is **not** driven by soil vs.
non-soil habitat contrasts. The Tier 2 significance does not replicate within soil
(primary p=0.011, soil-only p=0.626), suggesting the tier asymmetry may be specific
to the full cross-environment niche metric or may be underpowered in the soil-only subset.

**Files**: `data/genus_soil_levins_b.csv`, `data/pgls_input_soil_primary.csv`,
`data/pgls_soil_primary_result.csv`

In [ ]:
import pandas as pd
import numpy as np
import subprocess
from pathlib import Path
from scipy.stats import zscore

DATA = Path('../data')
R_BIN = '/home/hmacgregor/r_env/bin/Rscript'

SOIL_ENVS = {'soil', 'agricultural', 'farm', 'field', 'paddy', 'peatland', 'desert', 'shrub'}
print(f'Soil environment categories ({len(SOIL_ENVS)}): {sorted(SOIL_ENVS)}')

## Block 1 — Soil-restricted Levins' B

In [ ]:
env_mat = pd.read_csv(DATA / 'otu_env_matrix.csv')
env_tot = pd.read_csv(DATA / 'env_totals.csv')

print('env_mat columns:', env_mat.columns.tolist())
print('env_tot columns:', env_tot.columns.tolist())
print()
print('All Env_Level_1 categories in env_mat:')
print(env_mat['Env_Level_1'].value_counts())
print()
print('Available soil categories (intersection with SOIL_ENVS):')
print(set(env_mat['Env_Level_1'].unique()) & SOIL_ENVS)

In [ ]:
# Filter to soil environments
soil_mat = env_mat[env_mat['Env_Level_1'].isin(SOIL_ENVS)].copy()
soil_tot = env_tot[env_tot['Env_Level_1'].isin(SOIL_ENVS)].copy()

print(f'OTU × soil-env rows: {len(soil_mat):,}')
print(f'Unique OTUs in soil: {soil_mat["otu_id"].nunique():,}')
print(f'Soil env categories: {soil_mat["Env_Level_1"].nunique()}')
print(soil_tot.sort_values('Env_Level_1'))

In [ ]:
# Prevalence-adjusted detection rate: p_i = n_detections_i / n_total_samples_i
merged = soil_mat.merge(soil_tot, on='Env_Level_1')
merged['p_i'] = merged['n_samples_detected'] / merged['n_total_samples']

# Pivot to OTU × env matrix
pivot = merged.pivot_table(index='otu_id', columns='Env_Level_1',
                           values='p_i', fill_value=0.0)
print(f'Pivot shape: {pivot.shape} (OTUs × soil envs)')

# Row-normalise to get q_i = p_i / Σp_j
row_sums = pivot.sum(axis=1)
# Exclude OTUs with zero total detection across all soil envs
pivot = pivot[row_sums > 0]
row_sums = row_sums[row_sums > 0]
q = pivot.div(row_sums, axis=0)

# Levins' B and standardised B_std
B_soil = 1.0 / (q**2).sum(axis=1)
n_envs_detected = (pivot > 0).sum(axis=1)
n_soil_total = pivot.shape[1]
# Standardise using actual envs detected per OTU (mirrors NB02 formula)
B_soil_std = (B_soil - 1) / (n_envs_detected - 1).clip(lower=1)

otu_b = pd.DataFrame({
    'otu_id': pivot.index,
    'levins_B_soil': B_soil.values,
    'levins_B_soil_std': B_soil_std.values,
    'n_soil_envs_detected': n_envs_detected.values
})

print(f'OTUs with soil Levins B: {len(otu_b):,}')
print(f'B_soil_std range: {otu_b.levins_B_soil_std.min():.3f} – {otu_b.levins_B_soil_std.max():.3f}')
print(f'Mean B_soil_std: {otu_b.levins_B_soil_std.mean():.3f}')
print(f'OTUs in ≥2 soil envs: {(otu_b.n_soil_envs_detected >= 2).sum():,}')

## Block 2 — Aggregate to genus level

In [ ]:
# genus_trait_table has otu_genus mapping (or genus_lower)
traits = pd.read_csv(DATA / 'genus_trait_table.csv')
print('genus_trait_table columns:', traits.columns.tolist())
print(f'Rows: {len(traits):,}')
print('\nSample:')
print(traits.head(3).to_string())

In [ ]:
# otu_env_matrix uses 'otu_id'; genus_trait_table has 'otu_genus' as the OTU-level field
# Check if otu_id in env_mat matches any ID column in traits
sample_otu = env_mat['otu_id'].iloc[0]
print(f'Sample otu_id from env_mat: {sample_otu!r}')

# Check what ID columns traits has
for col in ['otu_genus', 'otu_id', 'genus_lower', 'genus']:
    if col in traits.columns:
        print(f'  traits.{col} sample: {traits[col].iloc[0]!r}')

In [ ]:
# Merge otu_b with genus mapping
# The otu_id in env_mat corresponds to otu_genus in genus_trait_table
otu_b['otu_id_str'] = otu_b['otu_id'].astype(str)
traits['otu_genus_str'] = traits['otu_genus'].astype(str)

otu_genus = otu_b.merge(
    traits[['otu_genus_str', 'genus_lower']].drop_duplicates(),
    left_on='otu_id_str', right_on='otu_genus_str', how='inner'
)
print(f'OTUs with genus mapping: {len(otu_genus):,}')

# Require OTU detected in ≥2 soil envs for stable estimate
otu_genus_filt = otu_genus[otu_genus['n_soil_envs_detected'] >= 2].copy()
print(f'After ≥2-soil-env filter: {len(otu_genus_filt):,}')

# Aggregate to genus (mean + count)
genus_soil = otu_genus_filt.groupby('genus_lower').agg(
    mean_B_soil_std=('levins_B_soil_std', 'mean'),
    n_otus=('otu_id', 'count'),
    mean_n_soil_envs=('n_soil_envs_detected', 'mean')
).reset_index()

# Require ≥5 OTUs for stable genus estimate
genus_soil_filt = genus_soil[genus_soil['n_otus'] >= 5].copy()
print(f'Genera with ≥5 OTUs in soil: {len(genus_soil_filt):,}')
genus_soil_filt.to_csv(DATA / 'genus_soil_levins_b.csv', index=False)
print('Saved: data/genus_soil_levins_b.csv')

## Block 3 — Per-Mb predictors (94-KO tiered list)

In [ ]:
gsize = pd.read_csv(DATA / 'genus_genome_size_gtdb.csv')
print('genome size cols:', gsize.columns.tolist())
gsize['genus_lower'] = gsize['genus_lower'].str.lower().str.strip()

# Merge trait table with genome sizes
genus_traits = traits.groupby('genus_lower').agg(
    mean_n_metal_types=('mean_n_metal_types', 'mean'),           # total 94-KO distinct metal types
    mean_n_defense_clusters=('mean_n_defense_clusters', 'mean'), # Tier 1 resistance
    mean_n_homeostasis_clusters=('mean_n_homeostasis_clusters', 'mean'), # Tier 2 homeostasis
    mean_n_metal_clusters=('mean_n_metal_clusters', 'mean')      # total cluster count
).reset_index()

genus_traits = genus_traits.merge(gsize[['genus_lower', 'mean_genome_size_bp']], on='genus_lower')
genus_traits['genome_mb'] = genus_traits['mean_genome_size_bp'] / 1e6

for col, out_col in [
    ('mean_n_metal_types', 'total_per_mb'),
    ('mean_n_defense_clusters', 'tier1_per_mb'),
    ('mean_n_homeostasis_clusters', 'tier2_per_mb')
]:
    genus_traits[out_col] = genus_traits[col] / genus_traits['genome_mb']

print(f'Genera with trait + genome data: {len(genus_traits):,}')
print(genus_traits[['genus_lower','total_per_mb','tier1_per_mb','tier2_per_mb']].describe())

In [ ]:
# Merge soil Levins' B with per-Mb predictors
merged = genus_soil_filt.merge(genus_traits, on='genus_lower')
print(f'Genera with soil B + traits: {len(merged):,}')

# Z-score predictors
merged = merged.dropna(subset=['mean_B_soil_std', 'total_per_mb', 'tier1_per_mb', 'tier2_per_mb'])
for col, z_col in [
    ('total_per_mb', 'ko_per_mb_total_z'),
    ('tier1_per_mb', 'ko_per_mb_tier1_z'),
    ('tier2_per_mb', 'ko_per_mb_tier2_z')
]:
    merged[z_col] = zscore(merged[col])

# Prepare PGLS input (column names must match pgls_mgnify_validation.R expectations)
pgls_input = merged[[
    'genus_lower', 'mean_B_soil_std',
    'ko_per_mb_total_z', 'ko_per_mb_tier1_z', 'ko_per_mb_tier2_z'
]].rename(columns={'mean_B_soil_std': 'biome_H_std'}).dropna()

out_path = DATA / 'pgls_input_soil_primary.csv'
pgls_input.to_csv(out_path, index=False)
print(f'PGLS input: {len(pgls_input)} genera → {out_path}')
print(pgls_input.describe())

## Block 4 — PGLS

In [ ]:
result = subprocess.run(
    [R_BIN, '../scripts/pgls_mgnify_validation.R',
     str(DATA / 'pgls_input_soil_primary.csv'),
     str(DATA / 'gtdb_bac_genus_pruned.tree'),
     str(DATA / 'pgls_soil_primary_result.csv')],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

In [ ]:
res = pd.read_csv(DATA / 'pgls_soil_primary_result.csv')
print(res.to_string())

## Block 5 — Comparison to primary analysis

In [ ]:
primary = {
    'Analysis': 'Primary (all envs, Levins\' B, n=997)',
    'total_beta': -0.022, 'total_p': 4e-7,
    'tier1_p': 0.256, 'tier2_p': 0.011
}

if res is not None:
    soil_total = res[res['predictor'].str.contains('total')].iloc[0]
    soil_tier1 = res[res['predictor'].str.contains('tier1')].iloc[0]
    soil_tier2 = res[res['predictor'].str.contains('tier2')].iloc[0]
    soil = {
        'Analysis': f'Soil-only (8 soil envs, Levins\' B_soil, n={int(soil_total.n_taxa)})',
        'total_beta': soil_total.beta, 'total_p': soil_total.p_value,
        'tier1_p': soil_tier1.p_value, 'tier2_p': soil_tier2.p_value
    }
else:
    soil = {'Analysis': 'Soil-only', 'total_beta': None, 'total_p': None, 'tier1_p': None, 'tier2_p': None}

comparison = pd.DataFrame([primary, soil])
print('\n=== Comparison ===')
print(comparison.to_string(index=False))
print()
print('Key question: Does β(total/Mb) remain negative and significant within soil only?')

In [ ]:
# Results already computed; display from CSV
import pandas as pd
from pathlib import Path
DATA = Path('../data')
res = pd.read_csv(DATA / 'pgls_soil_primary_result.csv')
print('=== Soil-restricted PGLS results ===')
print(res[['predictor','n_taxa','lambda','beta','SE','p_value']].to_string(index=False))
print()
primary_total_beta = -0.022; primary_total_p = 4e-7
soil_total = res[res['predictor'].str.contains('total')].iloc[0]
print(f'Primary:    β(total/Mb)={primary_total_beta:.3f}, p={primary_total_p:.1e}')
print(f'Soil-only:  β(total/Mb)={soil_total.beta:.3f}, p={soil_total.p_value:.5f}')
print()
print('Conclusion: Signal replicates within soil (β sign and magnitude preserved).')